# Pilot Test: Jupyter + Public LLM Memory Dump Feasibility

이 노트북은 사용자가 지정한 memory dump를 읽고, 로컬 분석 결과를 공개 LLM API에 연결하는 최소 동작 데모입니다.

In [ ]:
from pathlib import Path
import json
import os
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
WORKSPACE_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DUMP_PATH = ''  # 여기에 분석할 dump 경로를 직접 입력하세요
print('PROJECT_ROOT =', PROJECT_ROOT)
print('DUMP_PATH =', DUMP_PATH)
print('dump exists =', Path(DUMP_PATH).exists() if DUMP_PATH else False)
print('OPENAI_API_KEY configured =', bool(os.environ.get('OPENAI_API_KEY')))

## Imports and runtime options

In [ ]:
from analysis_pipeline import FeasibilityOptions, run_feasibility_analysis
from llm_assistant import LLMAssistant
from memory_kernel_analyzer import build_analysis_context, summarize_findings

MODEL = os.environ.get('OPENAI_MODEL', 'openai/gpt-oss-120b:free')
API_BASE = os.environ.get('OPENAI_API_BASE', 'https://openrouter.ai/api/v1')
ALLOW_RAW_CHUNK_EXCERPT = True
GENERATE_NEXT_STEPS = False

assistant = LLMAssistant(model=MODEL, api_base=API_BASE)
print('MODEL =', MODEL)
print('API_BASE =', API_BASE)
assistant.is_configured()

## Step 1. Build local analysis context

In [ ]:
context = build_analysis_context(DUMP_PATH)
summary = summarize_findings(context)
print(json.dumps(summary, ensure_ascii=False, indent=2)[:4000])

## Step 2. Inspect chunk samples sent to the LLM

In [ ]:
for sample in context.raw_chunk_samples[:3]:
    print(sample)
    print('-' * 80)

## Step 3. Run the feasibility pipeline

In [ ]:
options = FeasibilityOptions(
    dump_path=DUMP_PATH,
    allow_raw_chunk_excerpt=ALLOW_RAW_CHUNK_EXCERPT,
    raw_chunk_limit=2,
    generate_next_steps=GENERATE_NEXT_STEPS,
)
report = run_feasibility_analysis(DUMP_PATH, assistant, options)
report.llm_enabled

## Step 4. Review local findings

In [ ]:
print(json.dumps(report.findings, ensure_ascii=False, indent=2)[:4000])

## Step 5. Review LLM output

API 키가 없으면 이 셀은 `None` 또는 오류 메시지를 보여줍니다.

In [ ]:
print(report.llm_analysis)
print('\n' + '=' * 80 + '\n')
print(report.next_steps)

## Step 6. Script generation example

In [ ]:
if assistant.is_configured():
    script = assistant.generate_analysis_script(
        'memory dump에서 panic signal과 call trace 후보를 추출하는 Python 스크립트',
        findings=report.findings,
    )
    print(script)
else:
    print('OPENAI_API_KEY is not configured')